In [1]:
import os
import numpy as np
import librosa
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models

# ============================================================
# SETTINGS
# ============================================================

DATASET_DIR = r"D:/dataset/set"

SAMPLE_RATE = 16000
N_MFCC = 40
MAX_LEN = 200

LANGUAGES = ["Dogri", "English", "Hindi"]

# ============================================================
# MFCC EXTRACTION
# ============================================================

def extract_mfcc(file_path):

    audio, sr = librosa.load(
        file_path,
        sr=SAMPLE_RATE,
        mono=True
    )

    # Normalize
    audio = librosa.util.normalize(audio)

    # Remove silence
    audio, _ = librosa.effects.trim(audio, top_db=20)

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC
    )

    # Fix time dimension
    if mfcc.shape[1] < MAX_LEN:

        pad_width = MAX_LEN - mfcc.shape[1]

        mfcc = np.pad(
            mfcc,
            ((0, 0), (0, pad_width)),
            mode="constant"
        )

    else:

        mfcc = mfcc[:, :MAX_LEN]

    return mfcc


# ============================================================
# LOAD DATA
# ============================================================

X = []
y = []

for language in LANGUAGES:

    folder = os.path.join(DATASET_DIR, language)

    print("Loading:", language)

    for file in os.listdir(folder):

        if file.lower().endswith(".wav"):

            file_path = os.path.join(folder, file)

            try:

                mfcc = extract_mfcc(file_path)

                X.append(mfcc)
                y.append(language)

            except Exception as e:

                print("Error:", file, e)


X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("Labels:", y.shape)


# ============================================================
# ENCODE LABELS
# ============================================================

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print("Classes:", encoder.classes_)


# ============================================================
# ADD CHANNEL DIMENSION
# ============================================================

X = X[..., np.newaxis]

print("CNN input:", X.shape)


# ============================================================
# TRAIN / VALIDATION / TEST
# ============================================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


# ============================================================
# CNN MODEL
# ============================================================

model = models.Sequential([

    layers.Input(shape=(N_MFCC, MAX_LEN, 1)),

    layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.BatchNormalization(),

    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        64,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.BatchNormalization(),

    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(
        128,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.BatchNormalization(),

    layers.MaxPooling2D((2, 2)),

    layers.Dropout(0.3),

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.3),

    layers.Dense(
        len(LANGUAGES),
        activation="softmax"
    )
])


# ============================================================
# COMPILE
# ============================================================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# ============================================================
# TRAIN
# ============================================================

callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "language_cnn.keras",
        monitor="val_accuracy",
        save_best_only=True
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks
)


# ============================================================
# SAVE LABELS
# ============================================================

np.save(
    "language_classes.npy",
    encoder.classes_
)

print("Model saved successfully.")

Loading: Dogri


C:\Users\Lenovo\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading: English
Loading: Hindi
X shape: (30000, 40, 200)
Labels: (30000,)
Classes: ['Dogri' 'English' 'Hindi']
CNN input: (30000, 40, 200, 1)
Training: (21000, 40, 200, 1)
Validation: (4500, 40, 200, 1)
Testing: (4500, 40, 200, 1)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 40, 200, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 40, 200, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 100, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 20, 100, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 20, 100, 64)    │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 50, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 50, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 10, 50, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 5, 25, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 25, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,467 (431.51 KB)

 Trainable params: 110,019 (429.76 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 91s 131ms/step - accuracy: 0.9791 - loss: 0.0668 - val_accuracy: 0.9829 - val_loss: 0.0421
Epoch 2/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 86s 131ms/step - accuracy: 0.9954 - loss: 0.0164 - val_accuracy: 0.9996 - val_loss: 0.0022
Epoch 3/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 85s 130ms/step - accuracy: 0.9985 - loss: 0.0047 - val_accuracy: 0.9922 - val_loss: 0.0238
Epoch 4/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 85s 129ms/step - accuracy: 0.9973 - loss: 0.0091 - val_accuracy: 0.9862 - val_loss: 0.0397
Epoch 5/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 97s 147ms/step - accuracy: 0.9976 - loss: 0.0069 - val_accuracy: 0.9982 - val_loss: 0.0064
Epoch 6/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 105s 160ms/step - accuracy: 0.9992 - loss: 0.0031 - val_accuracy: 0.9998 - val_loss: 0.0010
Epoch 7/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 107s 163ms/step - accuracy: 0.9977 - loss: 0.0076 - val_accuracy: 0.9889 - val_loss: 0.0338
Epoch 8/30
657/657 ━━━━━━━━━━━━━━━━━━━━ 105s 160ms/step - accuracy: 0.9980 - loss